# FleetLogix — Avance 4: Arquitectura Cloud AWS

**Diseño conceptual, no desplegado en AWS.** Este notebook documenta los scripts proporcionados por Henry y la propuesta de arquitectura. No requiere credenciales ni crea recursos.


## 1. Arquitectura propuesta

Aplicaciones móviles → API Gateway → funciones Lambda → DynamoDB para estado operativo y SNS para alertas. S3 conserva históricos; PostgreSQL local se migraría a Amazon RDS y el ETL existente continuaría alimentando Snowflake. CloudWatch serviría para observabilidad y RDS tendría respaldos automáticos.

![Diagrama AWS](../Scripts/aws_architecture_diagram.png)


## 2. Funciones Lambda de referencia

`lambda_verificar_entrega`: consulta `deliveries_status` en DynamoDB, valida `delivery_id` y devuelve códigos 200, 400, 404 o 500.

`lambda_calcular_eta`: calcula una distancia aproximada con diferencias de coordenadas, estima tiempo con velocidad reportada y registra resultados en `vehicle_tracking`. **No considera la red vial ni el tráfico real.**

`lambda_alerta_desvio`: compara posición con puntos de referencia de `routes_waypoints`; si supera 5 km publica una alerta SNS y la guarda en `alerts_history`. **La distancia a puntos no equivale a distancia exacta a la ruta.**


## 3. Recursos que define `aws_setup.py`

- RDS PostgreSQL (`fleetlogix-db`), con retención de respaldos de 7 días y snapshot inicial.
- Bucket S3 `fleetlogix-data` con prefijos `raw-data/`, `processed-data/`, `backups/`, `logs/` y política de archivo a los 90 días. **La organización por fecha no está implementada en el script.**
- Cuatro tablas DynamoDB: `deliveries_status`, `vehicle_tracking`, `routes_waypoints`, `alerts_history`.
- Rol IAM `FleetLogixLambdaRole` con permisos amplios; en producción se recomienda mínimo privilegio.
- `migrar_datos_postgresql()` **solo escribe** `migrate_to_rds.sh`; no realiza la migración.


## 4. Orquestación propuesta

El script describe API Gateway POST `/deliveries/verify`, EventBridge cada cinco minutos para ETA y Kinesis para eventos GPS de desvíos. **Estos disparadores no se crean automáticamente con los scripts suministrados.** El API Gateway y su URL pública son puntos extra opcionales.


## 5. Seguridad, limitaciones y alcance

El archivo de configuración original contiene una contraseña incrustada y configura RDS con acceso público: **no se recomienda ejecutar sin corregirlo**. En un despliegue real usar Secrets Manager, subredes privadas, políticas IAM específicas, validación de entrada y gestión de errores. El ARN SNS es de ejemplo y debe reemplazarse por uno real.

Los recursos AWS no han sido desplegados ni probados; las cifras de costos dependen de región, fecha y uso.


## 6. Evidencia y archivos

- `Scripts/aws_architecture_diagram.png`: diagrama conceptual.
- `Scripts/lambda_functions/lambda_functions.py`: tres funciones proporcionadas por Henry.
- `Scripts/aws_setup.py`: configuración de recursos y generación de script de migración.
- `Documentacion/AWS_Análisis_Arquitectura.pdf`: explicación detallada solicitada en la guía del Avance 4.
